# Domain 2: Tool Design & MCP Integration
## Task Statement 2.2: Implement Structured Error Responses for MCP Tools
**Certification Weight**: 20% Exam Weight | **Standard**: Claude 4.5 & Extended Thinking

## 🎯 Exam Objective
Understand the Model Context Protocol (MCP) error contract, the `isError: true` flag, structured error taxonomy (`transient`, `validation`, `business`, `permission`), retryability signals, and the distinction between empty queries and execution failures.

---

## 🏗️ Architecture: MCP Structured Error Decision Flow

```mermaid
flowchart TD
    ToolFailure["Tool Execution Failure"] --> ClassifyError{"Classify Root Cause"}
    
    ClassifyError -->|"Network Timeout / 503"| Transient["category: transient<br>isRetryable: true<br>retryAfterMs: 1500"]
    ClassifyError -->|"Invalid Account Format"| Validation["category: validation<br>isRetryable: false<br>message: Expected ACC-XXXXX"]
    ClassifyError -->|"Over Daily Limit"| Business["category: business<br>isRetryable: false<br>message: Transfer exceeds limit"]
    ClassifyError -->|"403 Forbidden"| Permission["category: permission<br>isRetryable: false<br>message: Role admin required"]
    
    Transient --> MCPPayload["Format MCP tool_result:<br>isError: true<br>content: [structured error JSON]"]
    Validation --> MCPPayload
    Business --> MCPPayload
    Permission --> MCPPayload
    
    MCPPayload --> ClaudeRecovery{"Claude Recovery Loop"}
    ClaudeRecovery -->|"isRetryable: true"| BackoffRetry["Wait and Retry Tool Call"]
    ClaudeRecovery -->|"isRetryable: false"| AlternativeAction["Fix parameter / Escalate to Human"]
```

---

## ⚠️ Critical Exam Distinctions & Anti-Patterns

| Error Handling Pattern | Correct MCP Standard | Exam Trap to Reject |
|---|---|---|
| **Error Signaling Protocol** | Set `isError: true` on the `tool_result` and pass structured JSON detailing the failure reason. | **TRAP**: Throwing unhandled exceptions or returning HTTP 500 stack traces directly into the LLM context. |
| **Error Categorization** | Provide clear machine-readable taxonomy (`category: "transient" \| "validation" \| "business" \| "permission"`). | **TRAP**: Returning uniform generic text errors like `"Error: Operation failed"` which leaves Claude blind to whether a retry could succeed. |
| **Empty Results vs Failures** | Return successful `tool_result` with empty array `[]` when a search legitimately matches zero records. | **TRAP**: Setting `isError: true` when a database query finds 0 records, causing Claude to hallucinate system failure. |
| **Retry Coordination** | Explicitly include `isRetryable: boolean` to guide Claude's retry logic. | **TRAP**: Letting Claude blindly loop on permanent validation errors until iteration limits trigger. |

---

## 💡 Architectural Decision Matrix

| Failure Type | Category | `isRetryable` | Recommended Agent Action |
|---|---|---|---|
| **Downstream DB Timeout** | `transient` | `true` | Back off exponentially and re-execute. |
| **Invalid Regex Format** | `validation` | `false` | Re-format input parameters and retry with corrected payload. |
| **Insufficient Account Balance** | `business` | `false` | Halt transaction and explain policy limitation to customer. |
| **Missing Scope Token** | `permission` | `false` | Escalate to human supervisor; do not retry. |

---

---

### 🧠 Claude 4.5 Extended Thinking & Parameter Standards (October 2026)
- **Model Standard**: Use `claude-sonnet-4-5` or `claude-haiku-4-5`.
- **Temperature Omission**: When `thinking` is enabled (`thinking={"type": "enabled", "budget_tokens": ...}`), **do not provide `temperature`**, `top_p`, or `top_k`. The Anthropic API rejects custom sampling values with an HTTP `400 Bad Request` error.
- **Thinking Continuity**: In multi-turn loops, assistant reasoning blocks (`type: "thinking"`) must be returned **unmodified with their cryptographic signature** in subsequent turns. Stripping thinking blocks causes an API `400 Bad Request` error.
- **Case Facts Preservation**: Immutable transactional data (Order IDs, customer accounts, amounts) must be maintained in a dedicated Case Facts block outside conversation summaries.

In [ ]:
# Step 1: Environment & Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(" Environment initialized for TS 2.2 MCP Structured Errors.")

### 🚨 Step 2: Structured Error Helper & Tool Implementations
We implement `create_mcp_error` and tools demonstrating transient, validation, business, and empty result states.

In [ ]:
# Step 2: MCP Error Helper & Tools
def create_mcp_error(category: str, message: str, is_retryable: bool, details: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    """Formats an MCP structured error response conforming to certification standards."""
    return {
        "isError": True,
        "errorCategory": category,  # 'transient' | 'validation' | 'business' | 'permission'
        "isRetryable": is_retryable,
        "message": message,
        "details": details or {},
    }

def query_inventory(sku: str, simulate_mode: Optional[str] = None) -> Dict[str, Any]:
    """Simulates warehouse inventory lookups with structured error handling."""
    if simulate_mode == "timeout":
        return create_mcp_error("transient", "Warehouse database connection timed out after 3000ms.", is_retryable=True)
    if not sku.startswith("SKU-"):
        return create_mcp_error("validation", f"Invalid SKU format '{sku}'. Expected format: SKU-XXXX.", is_retryable=False)
    if sku == "SKU-99999":
        # Valid empty result: NOT an error!
        return {"isError": False, "sku": sku, "stock_count": 0, "is_empty_result": True}
    return {"isError": False, "sku": sku, "stock_count": 42, "is_empty_result": False}

def issue_refund(customer_id: str, amount: float) -> Dict[str, Any]:
    """Simulates refund processing with business policy bounds."""
    if amount > 500.0:
        return create_mcp_error(
            "business",
            f"Refund amount ${amount:.2f} exceeds automated policy threshold ($500.00). Escalate to manager.",
            is_retryable=False,
            details={"customer_id": customer_id, "threshold": 500.0, "requested": amount}
        )
    return {"isError": False, "customer_id": customer_id, "refund_amount": amount, "status": "success"}

print(" Structured error engine ready.")

### 🧪 Step 3: Interactive Simulation
We test all 4 primary error/result categories:
1. **Transient Timeout**: Retryable.
2. **Malformed SKU**: Non-retryable validation error.
3. **Valid Empty Result**: Success with 0 items (isError: False).
4. **Policy Cap Breach**: Business rule violation.

In [ ]:
# Step 3: Simulation
# 1. Transient Error
res_transient = query_inventory("SKU-101", simulate_mode="timeout")
print("Transient Error Result:", res_transient)

# 2. Validation Error
res_validation = query_inventory("INVALID_FORMAT_SKU")
print("\nValidation Error Result:", res_validation)

# 3. Valid Empty Result (isError: False!)
res_empty = query_inventory("SKU-99999")
print("\nValid Empty Result:", res_empty)

# 4. Business Rule Violation
res_business = issue_refund("CUST-100", 650.0)
print("\nBusiness Rule Error Result:", res_business)

### 🔬 Step 4: Verification Assertions
We assert that:
1. Transient errors have `isRetryable == True`.
2. Validation errors have `isRetryable == False`.
3. Valid empty results have `isError == False`.
4. Business errors have `isRetryable == False` and include details.

In [ ]:
# Step 4: Verification Checks
assert res_transient["isError"] is True
assert res_transient["errorCategory"] == "transient"
assert res_transient["isRetryable"] is True

assert res_validation["isError"] is True
assert res_validation["errorCategory"] == "validation"
assert res_validation["isRetryable"] is False

assert res_empty["isError"] is False
assert res_empty["stock_count"] == 0
assert res_empty["is_empty_result"] is True

assert res_business["isError"] is True
assert res_business["errorCategory"] == "business"
assert res_business["isRetryable"] is False

print(" All verification assertions passed! MCP error taxonomy verified.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Always return structured errors using `isError: true` with an explicit `errorCategory`.
- **Rule 2**: Mark `isRetryable: true` ONLY for transient infrastructure failures (timeouts, rate limits).
- **Rule 3**: Never return an error for valid empty query results (0 matches). Return `isError: false` and `stock_count: 0`.